# Лекция 05. Объекты, ссылки, копирование и JSON

Скопировали таблицу, поправили в копии одно число. Открыли исходную — там оно тоже поменялось. Чтобы разобраться, проследим, какие объекты создал Python и какие ссылки на них сохранил. Эта же модель объяснит, что происходит с аргументами функций, почему ошибка иногда оставляет после себя изменённый список и что удаётся сохранить в JSON.

## Цели

К концу лекции будем уметь проследить судьбу объекта: какие имена и контейнеры на него ссылаются, кто может его изменить и где понадобится копия. По дороге разберём равенство и идентичность, требования к ключам словаря и преобразование в JSON. В каждом случае проверяем своё объяснение коротким опытом.

## Перед началом

Списки, кортежи и словари нам уже знакомы по [лекции 2](../lesson02/lecture.ipynb). Понадобятся также функции и `import`. До запуска ячейки запишите прогноз. Если результат окажется другим, нарисуйте объекты и ссылки между ними.

На схемах две стрелки в один прямоугольник ведут к одному объекту. Адреса в памяти рисунок не показывает.

## Имена ведут к объектам

Выполним `a = [10, 20]`: появился список, с ним связано имя `a`. После `b = a` к этому списку можно обратиться ещё и через `b`. Второго списка пока никто не создавал. Он появится в строке `c = [10, 20]`: значения те же, объект другой.

У каждого объекта есть **тип**, **значение** и **идентичность**. Тип определяет допустимые операции, значение — содержимое, идентичность отличает объект от остальных. Перепривязка имени прежний объект не меняет.

![Имена a и b ведут к одному списку, c — к отдельному списку с такими же значениями.](assets/names-and-objects.svg)

### Как в одном списке помещаются разные типы

В `values = ["Аня", 20, True, None]` рядом оказались строка, число, логическое значение и `None`. Список хранит ссылки на них, поэтому общий тип элементам не требуется. `values[0]` возвращает строку с её методами, например `upper()`. У следующего элемента свои операции, а сам `values` остаётся списком.

![Четыре ссылки списка values ведут к объектам типов str, int, bool и NoneType.](assets/mixed-types.svg)

In [ ]:
a = [10, 20]
b = a
c = [10, 20]

print(type(a).__name__)
print(a == c, a is c)
print(a == b, a is b)

values = ["Аня", 20, True, None]
for value in values:
    print(repr(value), type(value).__name__)

## Равные значения и один объект

В опыте выше `a == c` даёт `True`: элементы списков равны. `a is c` даёт `False`: списков два. У `a` и `b` список общий, поэтому истинны обе проверки.

Для сравнения значений используем `==`, для проверки общей ссылки — `is`. `id(obj)` возвращает идентичность в виде числа, постоянного при жизни объекта. После его удаления число может достаться другому объекту.

Числа и строки сравнивайте через `==`: Python иногда переиспользует неизменяемые объекты, и полагаться на это в программе нельзя. Для `None` обычно пишем `value is None`: объект, обозначающий отсутствие значения, один. В фильтре ниже отбрасываем только `None`. Нули сохранятся: измерения есть, просто получились нулевыми.

In [ ]:
measurements = [0, None, 12, 0, None]
present = [value for value in measurements if value is not None]
print(present)

## Изменить список или перепривязать имя

Пусть `a` и `b` ссылаются на один список. Вызов `b.append(30)` добавит в него элемент, и через оба имени мы увидим `[10, 20, 30]`.

При `b = [99]` происходит другое: создаётся новый список, и имя `b` переходит к нему. Имя `a` по-прежнему связано со старым. Такая смена связи называется перепривязкой; старый список остаётся прежним.

### А если написать `a[:] = []`?

Здесь заменяется содержимое всего списка. Список остаётся прежним, поэтому через `b` тоже будет видна пустота. Того же результата можно добиться вызовом `a.clear()` или командой `del a[:]`.

У среза получились две роли: выражение `a[:]` создаёт поверхностную копию, а присваивание `a[:] = ...` меняет исходный список. В трёх опытах ниже одинаковое начало. Проследите, что меняется дальше.

In [ ]:
a = [10, 20]
b = a
b.append(30)
print("Изменение:", a, b, a is b)

a = [10, 20]
b = a
b = [99]
print("Перепривязка:", a, b, a is b)

a = [10, 20]
b = a
a[:] = []
print("Очистка:", a, b, a is b)

## Почему `+=` и `+` дают разные последствия

Со списком запись `a += [30]` добавляет элемент в существующий объект. Все ссылки на этот список увидят добавку. В записи `a = a + [30]` сложение создаёт новый список, после чего имя `a` связывается с результатом. Остальные ссылки остаются у прежнего списка.

У кортежа содержимое изменить нельзя, поэтому `t += (3,)` создаёт новый кортеж и перепривязывает `t`. Старый кортеж при этом остаётся прежним. Значок `+=` один, а последствия зависят от типа объекта.

In [ ]:
a = [10, 20]
b = a
a += [30]
print(a, b, a is b)

a = [10, 20]
b = a
a = a + [30]
print(a, b, a is b)

t = (1, 2)
old = t
t += (3,)
print(t, old, t is old)

## Одно имя снаружи и внутри функции

В следующем примере и параметр функции, и внешняя переменная называются `lst`. Это два разных имени в разных областях видимости. Посмотрим, что произойдёт, если внутри функции присвоить параметру пустой список.

In [ ]:
def change_list(lst):
    lst = []
    print(len(lst))


lst = [1, 2, 3]
change_list(lst)          # Внутри функции печатается 0.
assert lst == [1, 2, 3]
print(lst)                # Снаружи остался [1, 2, 3].

При вызове функции оба имени `lst` сначала ссылаются на один список `[1, 2, 3]`. Строка `lst = []` создаёт новый пустой список и связывает с ним **только локальное имя — параметр функции**. Поэтому внутри `len(lst)` равно нулю.

Внешнее имя `lst` всё это время продолжает ссылаться на исходный список. Он не менялся, поэтому проверка после вызова проходит. Одинаковое написание имён не делает их одной переменной.

А теперь изменим сам переданный объект. Вызов `lst.clear()` очищает существующий список. Через внешнее имя будет виден тот же, теперь пустой список: эту перемену возврат из функции не отменяет.

In [ ]:
def clear_list(lst):
    lst.clear()
    print(len(lst))


lst = [1, 2, 3]
clear_list(lst)           # Внутри функции тоже печатается 0.
assert lst == []
print(lst)                # Но теперь снаружи тоже [].

## Что именно нельзя менять в кортеже

Соберём запись `("Аня", [10, 20])`. Кортеж хранит две ссылки: на строку с именем и на список баллов. Подставить вместо второго элемента другой список нельзя. А добавить балл в уже существующий список можно — ссылка в кортеже останется той же.

В примере имя `scores` и второй элемент `record` ведут к одному списку. После `scores.append(30)` новый балл виден и через кортеж. Вложенный объект доступен из обоих мест.

In [ ]:
scores = [10, 20]
record = ("Аня", scores)
scores.append(30)
print(record)
print(record[1] is scores)

# Намеренная ошибка: заменить элемент кортежа нельзя.
try:
    record[1] = [99]
except TypeError as error:
    print(type(error).__name__)

## Две строки таблицы оказались одной

Создадим таблицу `[[0] * 3] * 2` и заменим один ноль семёркой. При проверке она обнаружится сразу в обеих строках.

Внутренний список был создан один раз. Умножение повторило ссылку на него, поэтому `wrong[0]` и `wrong[1]` ведут к одному объекту. В списковом включении выражение `[0] * 3` выполняется заново на каждой итерации и создаёт отдельную строку.

![Умножение повторяет ссылку на одну строку; списковое включение создаёт две отдельные строки. Показано состояние после изменения элемента.](assets/repeated-rows.svg)

Внутри `[0] * 3` тоже повторяются ссылки — на неизменяемое число. Когда присваиваем элементу списка `7`, мы заменяем одну ссылку. Сам объект `0` остаётся прежним.

In [ ]:
wrong = [[0] * 3] * 2
wrong[0][1] = 7
print(wrong, wrong[0] is wrong[1])

right = [[0] * 3 for _ in range(2)]
right[0][1] = 7
print(right, right[0] is right[1])

## Что копирует `.copy()`

Вернёмся к таблице, которую хотели исправить независимо от исходной. `base.copy()` создаёт новый внешний список: добавленная в него строка у `base` не появится. Существующие строки пока общие, поэтому замена элемента внутри такой строки будет видна с обеих сторон.

Такую копию называют **поверхностной**. Срез списка `base[:]` работает так же. У `dict.copy()` появляется новый словарь, который хранит прежние ссылки на ключи и значения.

![У base и copied разные внешние списки, но их элементы ведут к одним и тем же вложенным спискам.](assets/shallow-copy.svg)

In [ ]:
base = [[1, 2], [3, 4]]
copied = base.copy()
print(copied is base, copied[0] is base[0])

copied.append([5, 6])
copied[0][0] = 99
print("Исходный:", base)
print("Копия:", copied)

## Копируем столько, сколько понадобится

В нашей таблице лежат строки, а в строках — числа. Создадим новый внешний список и отдельно скопируем каждую строку. После этого элементы можно заменять независимо от исходной таблицы. Копировать сами числа незачем: изменить их на месте всё равно нельзя.

Способ годится, пока мы знаем форму данных. Если внутри строки окажется ещё один список, он останется общим. Перед копированием полезно определить, какие именно объекты программа собирается менять.

In [ ]:
base = [[1, 2], [3, 4]]
independent = [row.copy() for row in base]
independent[0][0] = 99
print(base)
print(independent)
print(independent[0] is base[0])

## Глубокая копия и общие ссылки

Когда вложенных уровней много, их может обойти `copy.deepcopy`. Для наших списков и словарей он создаст новые контейнеры, сохранив связи между ними.

В следующем примере одна строка попала в `base` дважды. После глубокого копирования у `cloned` будет своя строка, но обе его ссылки по-прежнему будут вести к ней. Изменение через `cloned[0]` обнаружится и в `cloned[1]`; исходную `row` оно уже не затронет.

![Исходный список и глубокая копия имеют свои внутренние строки; две ссылки внутри каждого списка по-прежнему ведут к одной строке.](assets/deepcopy-graph.svg)

На такой обход расходуются время и память. Плоскому списку чисел достаточно `.copy()`, а в большой структуре иногда нужно скопировать только ту часть, которую будем менять. Неизменяемые объекты `deepcopy` может использовать повторно.

In [ ]:
from copy import deepcopy

row = [1, 2]
base = [row, row]
cloned = deepcopy(base)

print(cloned is base)
print(cloned[0] is base[0])
print(cloned[0] is cloned[1])
cloned[0].append(3)
print(base)
print(cloned)

## Что требуется от ключа словаря

Кортеж со списком внутри получился неизменяемым снаружи, но в ключи словаря он всё равно не годится. От ключа требуется **хешируемость**: его хеш должен оставаться постоянным в течение жизни объекта, а сам объект должен поддерживать сравнение. У равных хешируемых объектов хеши совпадают. Обратное неверно: разные значения тоже могут получить один хеш.

`list`, `dict` и `set` не хешируются. Кортеж хешируется, если это умеют все его элементы; вложенный список нарушает условие.

Есть ещё следствие равенства, которое легко пропустить. `bool` — подтип `int`, поэтому `True == 1 == 1.0`, и хеши у них одинаковые. В словаре три такие записи займут одно место: очередное значение заменит предыдущее. Если в задаче существенен тип, его придётся проверить отдельно. Устройство самой хеш-таблицы оставим до лекции 7.

In [ ]:
print(isinstance(True, int), type(True) is int)
print(True == 1 == 1.0)
print(hash(True) == hash(1) == hash(1.0))
labels = {True: "да", 1: "один", 1.0: "один с точкой"}
print(labels, len(labels))

# Намеренная ошибка: внутри кортежа лежит нехешируемый список.
try:
    hash(("Аня", [10, 20]))
except TypeError as error:
    print(type(error).__name__)

## JSON: данные выходят из программы

До сих пор наши списки и ссылки жили в памяти программы. Теперь подготовим запись об Ане для сохранения в файл или передачи другой программе. Получателю понадобится понятное представление данных.

**JSON** — один из таких текстовых форматов. Преобразование данных для хранения или передачи называется **сериализацией**. Функция `json.dumps` создаёт строку с JSON, а `json.loads` читает её и создаёт объекты Python. Буква `s` здесь от *string*; модуль `json` входит в стандартную библиотеку.

![dumps превращает словарь Python в текст JSON, loads создаёт по этому тексту новый словарь. True и None записываются как true и null.](assets/json-roundtrip.svg)

В примере `ensure_ascii=False` оставляет русские буквы читаемыми, а `indent=2` добавляет отступы. Это оформление текста. Переменная `text` будет содержать строку; словарь снова появится после `loads`.

In [ ]:
import json

profile = {"name": "Аня", "scores": [10, 20], "active": True, "note": None}
text = json.dumps(profile, ensure_ascii=False, indent=2)
print(text)
print(type(text).__name__)

restored = json.loads(text)
print(restored["name"], restored["scores"])
print(restored == profile, restored is profile)

## Что получится после чтения

У JSON свой набор типов. Кортеж после чтения станет списком: для обоих в JSON есть только массив.

| В Python | В JSON | После `loads` |
| --- | --- | --- |
| `dict` со строковыми ключами | object: `{...}` | `dict` |
| `list` или `tuple` | array: `[...]` | `list` |
| `str` | строка в двойных кавычках | `str` |
| `int`, конечный `float` | число | `int` или `float` |
| `True`, `False`, `None` | `true`, `false`, `null` | `True`, `False`, `None` |

Ключи в JSON — строки. `dumps` принимает и некоторые другие ключи, например число `7`, но записывает его как `"7"`. Обратно число не восстановится; для обмена сразу выбирайте строковые ключи.

`set` и `bytes` модуль по умолчанию не преобразует. Их представление придётся выбрать самим и согласовать с получателем: например, передать набор списком.

У текста тоже есть правила: двойные кавычки, никаких комментариев и запятой после последнего элемента. `str(some_dict)` даёт представление Python, поэтому вместо `json.dumps` не подходит.

In [ ]:
source = {7: (10, 20)}
text = json.dumps(source)
restored = json.loads(text)
print(text)
print(restored)
print(restored == source)
print(type(list(restored)[0]).__name__, type(restored["7"]).__name__)

# Намеренная ошибка: множество не входит в поддерживаемые типы.
try:
    json.dumps({"tags": {"python", "books"}})
except TypeError as error:
    print(type(error).__name__)

## Строка или открытый файл

Теперь запишем данные в файл. Функции почти те же: убираем `s` из названия и передаём открытый файловый объект.

| Функция | Откуда → куда |
| --- | --- |
| `json.dumps(data)` | объекты → строка |
| `json.loads(text)` | строка → объекты |
| `json.dump(data, file)` | объекты → открытый файл |
| `json.load(file)` | открытый файл → объекты |

Вызов `open(path, "w", encoding="utf-8")` создаёт текстовый файл или заменяет прежнее содержимое. Режим `"r"` открывает его для чтения, а `with` закрывает при выходе из блока, в том числе после ошибки. `dump` и `load` получают объект `file`; путь мы уже передали в `open`.

Для опыта возьмём временную папку: `TemporaryDirectory` удалит её после внешнего `with`. Для постоянного хранения нужен обычный путь. Файл содержит один JSON-документ; несколько записей можно собрать в список.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

profile = {"name": "Аня", "scores": [10, 20], "active": True, "note": None}

with TemporaryDirectory() as directory:
    path = Path(directory) / "profile.json"
    with open(path, "w", encoding="utf-8") as file:
        json.dump(profile, file, ensure_ascii=False, indent=2)

    with open(path, "r", encoding="utf-8") as file:
        restored = json.load(file)
    print(restored == profile)

## Что стало с общей ссылкой

В `base` два элемента ведут к одной строке. Мы уже проверили, что `deepcopy` сохраняет эту связь внутри копии. В JSON содержимое списка будет записано дважды, а при чтении появятся два отдельных списка. У формата нет отметки «здесь тот же объект, что и выше».

Поэтому после `json.loads(json.dumps(data))` могут измениться и типы, и связи. Такая запись годится для обмена в рамках JSON, но точного повторения произвольной структуры Python от неё ждать нельзя.

In [ ]:
row = [1, 2]
base = [row, row]
cloned = deepcopy(base)
restored = json.loads(json.dumps(base))

print(cloned[0] is cloned[1])
print(restored[0] is restored[1])
restored[0].append(3)
print(restored)

## JSON прочитан. Что внутри?

Если обрамить ключи одинарными кавычками, получим `json.JSONDecodeError`. А документ `{"scores": "десять"}` прочитается без ошибок. Мы получим словарь со строкой там, где собирались складывать баллы.

После `loads` проверяем, что пришло: внешний тип, нужные поля, их типы и допустимые значения. JSON может содержать даже одно число или `null`. Успешное чтение подтверждает, что текст удалось разобрать; пригодность данных для нашей задачи ещё предстоит проверить.

In [ ]:
# Намеренная ошибка: одинарные кавычки — синтаксис Python, не JSON.
try:
    json.loads("{'scores': [10, 20]}")
except json.JSONDecodeError as error:
    print(type(error).__name__)

data = json.loads('{"scores": "десять"}')
print(isinstance(data, dict))
print(isinstance(data.get("scores"), list))

## Неожиданно, но по правилам

Вернёмся к кортежу с именем и баллами. Раньше мы добавляли балл через `append`, теперь попробуем `record[1] += [30]`. Обычное `t += (3,)` уже работало без ошибок.

До запуска запишите, чего ждёте от этой строки. Если будет ошибка, каким после неё окажется список баллов?

In [ ]:
scores = [10, 20]
record = ("Аня", scores)

# Намеренная ошибка; перехватываем её, чтобы увидеть состояние после операции.
try:
    record[1] += [30]
except TypeError as error:
    print(type(error).__name__)

print(record)
print(scores)
print(record[1] is scores)

Получили `TypeError`, но число `30` уже в списке. Чтобы понять результат, пройдём операцию по шагам.

Сначала Python получает список из `record[1]`. Затем список выполняет `+= [30]`: меняется на месте и возвращает себя как результат операции. Остаётся записать результат обратно в `record[1]`. Здесь и возникает ошибка — присваивать элементу кортежа нельзя.

Список успел измениться, а кортеж сохранил прежние ссылки. Исключение не откатывает действия, выполненные до него, даже в пределах одной строки.

У `record[1].append(30)` присваивания обратно нет, поэтому всё проходит спокойно. В записи `record[1] = record[1] + [30]` сложение создаст новый список. Записать его в элемент кортежа тоже не удастся, зато старый вложенный список останется прежним.

## Проверим себя

Ответьте без запуска; если запутались, нарисуйте объекты и ссылки.

1. В списке `values = ["Аня", 20, True, None]` четыре разных типа. Что именно хранит список?
2. После `b = a` выполняем `a = []` или `a[:] = []`. Что увидим через `b`?
3. Функция получила список, выполнила `items.append(1)`, а затем `items = []`. Что увидит вызывающий код после возврата?
4. Почему `t += (3,)` работает, а `record[1] += [30]` меняет список и затем вызывает ошибку?
5. Какие изменения после `copied = base.copy()` затронут исходные данные? Сделает ли `deepcopy` общие строки независимыми внутри копии?
6. Что передают в `loads` и `load`? Сохранит ли JSON кортеж, числовой ключ и общую ссылку?

<details>
<summary>Ответы для проверки</summary>

1. Ссылки на объекты. Тип принадлежит каждому из них.
2. При присваивании `b` останется у прежнего списка; замена среза очистит общий объект.
3. В исходном списке останется добавленный элемент. Перепривязка локального параметра не изменила внешнее имя; при возврате ничего не откатывается.
4. В первом случае перепривязывается имя. Во втором список меняется до запрещённого присваивания элементу кортежа; отката нет.
5. Изменения общих вложенных объектов. `deepcopy` отделит их от оригинала, но сохранит общие ссылки внутри копии.
6. `loads` читает строку, `load` — открытый файл. Кортеж станет списком, числовой ключ — строкой, повторённый список восстановится отдельными объектами.

</details>

## Итоги

В этой лекции мы следили за объектами и ссылками на них. Имя можно перепривязать, объект — изменить, а контейнер может хранить ссылку на тот же объект, который уже доступен из другого места. При передаче аргумента локальное имя параметра связывается с тем же объектом. Его изменения видны через общие ссылки, а перепривязка параметра внешних имён не затрагивает. Так же очистка срезом видна через другие имена, а неизменяемый кортеж может содержать изменяемый список.

При копировании смотрим, какие объекты должны стать отдельными. Поверхностная копия создаёт внешний контейнер; глубокая обходит вложенные и сохраняет связи внутри результата. JSON сохраняет данные по правилам своего формата, поэтому после чтения проверяем типы и содержимое.

На [семинаре](seminar.ipynb) применим эту модель к небольшим программам, затем перейдём к [домашней работе](tasks.md). Ниже — ещё три следствия тех же правил: циклы, сборка мусора и общий аргумент по умолчанию.

## Дополнительно: циклы и `memo`

Список может содержать ссылку на самого себя. Если просто следовать по вложенным ссылкам, будем снова и снова возвращаться к нему.

`deepcopy` ведёт служебный словарь `memo`: для уже встреченного объекта там записана его копия. Сначала создаётся пустой список и запоминается соответствие, затем копируются элементы. При повторной встрече исходного списка берётся уже созданная копия. Ссылка замыкается внутри нового графа, и обход заканчивается.

Этот же механизм сохранял общую строку в предыдущих опытах. JSON представить цикл не может: по умолчанию `json.dumps` обнаружит его и вызовет `ValueError`.

In [ ]:
loop = []
loop.append(loop)
cloned_loop = deepcopy(loop)
print(cloned_loop is loop)
print(cloned_loop[0] is cloned_loop)

# Намеренная ошибка: JSON не хранит циклические ссылки.
try:
    json.dumps(loop)
except ValueError as error:
    print(type(error).__name__, error)

## Дополнительно: когда объект становится мусором

До сих пор мы создавали объекты и меняли ссылки. Посмотрим, что происходит, когда ссылки исчезают. Команда `del a` удаляет имя, но через `b` список всё ещё доступен:

```python
a = [10, 20]
b = a
del a
print(b)  # [10, 20]
del b
```

После последней строки этого отдельного примера до списка уже не добраться, и его память можно освободить. При `a[:] = []` сам список остаётся доступным, зато могут освободиться его прежние элементы — если других ссылок на них нет.

В CPython используется подсчёт ссылок. У недостижимого цикла внутренние ссылки сохраняются, поэтому такие циклы обнаруживает дополнительный сборщик мусора. Точный момент освобождения язык Python не обещает. Память может остаться у интерпретатора для следующих объектов, а файлы мы закрываем через `with`, не дожидаясь сборки мусора.

## Дополнительно: откуда берётся общий аргумент по умолчанию

В `def collect(item, items=[])` список создаётся при выполнении определения функции. При следующих вызовах без второго аргумента используется тот же объект. В первом вызове туда попадёт чай, во втором — кофе. Первый результат тоже изменится: он всё ещё ссылается на этот список.

Если по умолчанию задать `None`, новый список можно создавать внутри вызова. Проверка `items is None` заодно сохраняет связь с явно переданным пустым списком: функция будет работать именно с ним.

In [ ]:
def collect(item, items=[]):
    items.append(item)
    return items


first = collect("чай")
second = collect("кофе")
print(first, second, first is second)


def collect_separately(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items


first = collect_separately("чай")
second = collect_separately("кофе")
print(first, second, first is second)

## Полезные материалы

- [Модель данных: объекты, значения и типы](https://docs.python.org/3.14/reference/datamodel.html#objects-values-and-types).
- [Поверхностное и глубокое копирование](https://docs.python.org/3.14/library/copy.html).
- [Хешируемость](https://docs.python.org/3.14/glossary.html#term-hashable).
- [JSON: преобразования, строки и файлы](https://docs.python.org/3.14/library/json.html).
- [Почему `+=` внутри кортежа меняет список и вызывает ошибку](https://docs.python.org/3.14/faq/programming.html#why-does-a-tuple-i-item-raise-an-exception-when-the-addition-works).
- [Сборщик мусора CPython](https://docs.python.org/3.14/library/gc.html).
- [Почему изменяемые значения по умолчанию общие](https://docs.python.org/3.14/faq/programming.html#why-are-default-values-shared-between-objects).
- [Как Python передаёт аргументы в функцию](https://docs.python.org/3.14/faq/programming.html#how-do-i-write-a-function-with-output-parameters-call-by-reference).